# 03 · Reconciling grid forecasts to country totals

**What this teaches:** how `views_frames_reconcile` makes grid (`pgm`) predictions sum, per posterior draw, to their country (`cm`) totals — the per-draw top-down proportional method, the frame-level orchestrator, and the conservation / zero-preservation / joint-sampling properties — with plots (and ideally an animation) that make the rescaling visible.

**Audience:** anyone wiring reconciliation into a forecast pipeline, or wanting to *see* what the algorithm does to a distribution.

> **🚧 ROADMAP — draft, not yet built.** This notebook currently holds only the *plan* (markdown). The section outline below is what we align on **before** the first content pass; code cells get filled in afterwards. Edit these markdown cells freely — this is the working document for the notebook's design.

## 📋 Roadmap (section-by-section)

**1. The problem.** Independently-forecast grid cells don't sum to the country total. Country totals are authoritative; we want the grid cells rescaled to honour them — *per draw*, preserving each cell's relative share and its zeros. Why not other methods (the principled probabilistic upgrade is future work, views-postprocessing C-37).

**2. The leaf operation — `reconcile_proportional(grid, country)`.** One country, both point (1-D) and probabilistic (S, N). Show the per-draw proportional rescaling; **zeros stay zero**; non-negativity; the **all-zero-draw edge** (stays zero, not forced to the total).

**3. Make it visible (the plots / animation).** Before→after bar charts of the grid cells; the conservation line (`sum == country`); ideally an **animation** sweeping over draws (or over the scaling factor). *(The animation was prototyped in an earlier exploration; we re-create its logic in-notebook — self-contained, nothing referenced externally.)*

**4. The frame orchestrator — `ReconciliationModule(map_keys, map_vals).reconcile(cm, pgm)`.** Multi-country, multi-month; the **injected** `(time, priogrid) → country` mapping (never fetched); de-mutation (returns a *new* pgm frame); fail-loud validation (level / sample-count / time-coverage / missing-country guards).

**5. The contract properties.** `assert_reconcile_contract` — sum-to-country per draw, zero-preservation, non-negativity, level correctness, and the **mapping-injected** sensitivity check (permute the mapping → the result changes ⇒ it's used, not fetched).

**6. Joint sampling & subadditivity.** Why the regional worst-case is computed on the *aggregated* draws, not summed per-cell — `expected_shortfall(aggregate) ≠ Σ expected_shortfall(cell)` (subadditive). Bridges to notebook 02's `expected_shortfall` + `aggregate_distributions`.

**7. Provenance coda.** The numpy reconciler is **bit-identical** to the frozen views-reporting torch oracle (the `scripts/verify_reconcile_parity.py --oracle` story); the WET relocation (Epic 11 / ADR-023).

## 🔬 Additions from the expert-method-review panel (2026-06-26)

*Folded in from the practitioner panel (esp. Hyndman + Wilke/Kay + VIEWS-FAO). These slot into the roadmap above. Tracked as register C-60/C-61.*

- **NEW §A — Situate proportional reconciliation in its literature, and separate implementation-fidelity from method-quality (C-60).** Name the lineage: per-draw **top-down proportional** is the pragmatic, information-losing baseline; **MinT** (Wickramasuriya2019) is the optimal linear reconciliation; **probabilistic reconciliation** (Panagiotelis2023) is the principled draw-level version; the principled upgrade here is deferred (views-postprocessing C-37). And state plainly: **bit-identity to the torch oracle proves the *port* is faithful, not that the *method* is good** — don't let the parity story read as methodological endorsement.
- **NEW §B — Does reconciliation help?** Beyond *conserving* the country total, show its effect on a metric: on synthetic data with a known country truth, compare the country-aggregate's **coverage/skill before vs after** reconciling the grid.
- **NEW §C — Spatial / map view on a toy lattice (C-61).** Show the grid→country rescaling on a synthetic square lattice (a small map before/after), so the spatial effect is visible. Toy geography only (ADR-014).
- **NEW §D — Decision relevance.** The reconciled, coherent country-level `expected_shortfall` (the FAO-facing `severe_scenario`) / onset as the planning-relevant outputs. *(Bernardo1979; VIEWS-FAO consumer.)*

## 🎨 Source material — self-contained (logic ported in, nothing referenced externally)

This notebook **regenerates its own data, plots, and animation from code**; it depends on no
file outside this repo.

- *In-repo to adapt:* `tests/fixtures/reconciliation_*.npz` (a realistic cm/pgm example if we want one beyond the toy case); `scripts/verify_reconcile_parity.py` (drift / conservation logic); `tests/test_reconcile_conformance.py` (the synthetic scenario builder + the injected-mapping sensitivity pattern).
- The reconciliation plots/animation were prototyped earlier; during the build we **port that logic into this notebook** (copied in, self-contained). We do not reference any notebook outside the repo — if there's prior art worth keeping, we lift what we need into here.

## ❓ Open questions / decisions

1. **How much animation to re-create** — a faithful re-creation of the prior prototype, or a simplified static sequence? (`matplotlib.animation` renders to GIF/HTML but bloats the notebook; a small stills strip is lighter and GitHub-renderable.)
2. **Synthetic geography realism** — a toy 2-country / handful-of-cells example for clarity, vs the frozen-fixture realistic mapping for credibility. (Leaning: toy first, then one realistic.)
3. Whether to show the **`severe_scenario` aggregation** here or keep it in notebook 02 and just cross-link.

## ✅ Conventions for these notebooks

- **Public, frozen API only.** Everything shown uses the published `views_frames` / `views_frames_summarize` / `views_frames_reconcile` surface — so the notebook doubles as a living contract demo. If a cell *wants* a convenience that doesn't exist (e.g. `frame.to_parquet`), that's a **demand signal** to record, not a reason to reach into the leaf (see register D-11).
- **Synthetic data only.** No `viewser`, no domain fetches, no `views_*` consumer imports — every array is generated in-notebook with a fixed seed. Zero domain knowledge (ADR-001).
- **Un-gated dev artifact.** Lives in `notebooks/` (like `research/` and `scripts/`), outside `src/`; the package never imports it, so the import-DAG and the frozen core are untouched. Plotting/jupyter deps are an **optional extra**, never runtime deps.
- **Reproducible + light.** Fixed seeds; small sample sizes that render fast; no heavyweight state. A reader should be able to *Run All* in well under a minute.